In [1]:
import scanpy as sc
import pandas as pd

In [2]:
from pathlib import Path
import os, sys

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if Path.cwd().resolve() != PROJECT_ROOT:
    os.chdir(PROJECT_ROOT)

root_str = str(PROJECT_ROOT)
if root_str in sys.path:
    sys.path.remove(root_str)
sys.path.insert(0, root_str)

#print("PROJECT_ROOT =", PROJECT_ROOT)
#print("cwd =", Path.cwd())

In [3]:
filePath = './data/raw/Parse_10M_PBMC_cytokines_Donor1_10per.h5ad'
adata = sc.read_h5ad(filePath)

In [4]:
control_key = "is_control"
condition_keys = "cytokine"
control_name = "PBS"
condition_rep_keys = "gene_embeddings"
random_seed = 42
dataset_name = "PBMC"

In [5]:
gene_fix_map = {
    # CSF 家族
    'M-CSF': 'CSF1',
    'G-CSF': 'CSF3',
    'GM-CSF': 'CSF2',
    
    # 干扰素 (IFN) 家族
    'IFN-gamma': 'IFNG',
    'IFN-alpha1': 'IFNA1',
    'IFN-beta': 'IFNB1',
    'IFN-epsilon': 'IFNE',
    'IFN-omega': 'IFNW1',
    'IFN-lambda1': 'IFNL1',
    'IFN-lambda2': 'IFNL2',
    'IFN-lambda3': 'IFNL3',
    
    # 白介素 (IL) 家族 - 去掉横杠并修正别名
    'IL-1-alpha': 'IL1A',
    'IL-1-beta': 'IL1B',
    'IL-1Ra': 'IL1RN',
    'IL-2': 'IL2',
    'IL-3': 'IL3',
    'IL-4': 'IL4',
    'IL-5': 'IL5',
    'IL-6': 'IL6',
    'IL-7': 'IL7',
    'IL-8': 'CXCL8',  # IL-8 的标准基因名是 CXCL8
    'IL-9': 'IL9',
    'IL-10': 'IL10',
    'IL-11': 'IL11',
    'IL-12': 'IL12A', # 异源二聚体，映射到A亚基
    'IL-13': 'IL13',
    'IL-15': 'IL15',
    'IL-16': 'IL16',
    'IL-17A': 'IL17A',
    'IL-17B': 'IL17B',
    'IL-17C': 'IL17C',
    'IL-17D': 'IL17D',
    'IL-17E': 'IL25', # IL-17E 的标准基因名是 IL25
    'IL-17F': 'IL17F',
    'IL-18': 'IL18',
    'IL-19': 'IL19',
    'IL-20': 'IL20',
    'IL-21': 'IL21',
    'IL-22': 'IL22',
    'IL-23': 'IL23A', # 异源二聚体，映射到A亚基
    'IL-24': 'IL24',
    'IL-26': 'IL26',
    'IL-27': 'IL27',
    'IL-31': 'IL31',
    'IL-32-beta': 'IL32', # beta 是剪接变体，基因还是 IL32
    'IL-33': 'IL33',
    'IL-34': 'IL34',
    'IL-35': 'EBI3',  # 异源二聚体 (IL12A + EBI3)，映射到 EBI3
    'IL-36-alpha': 'IL36A',
    'IL-36Ra': 'IL36RN',
    
    # 肿瘤坏死因子 (TNF) 超家族
    'TNF-alpha': 'TNF',
    'TRAIL': 'TNFSF10',
    'TWEAK': 'TNFSF12',
    'APRIL': 'TNFSF13',
    'BAFF': 'TNFSF13B',
    'LIGHT': 'TNFSF14',
    'TL1A': 'TNFSF15',
    'GITRL': 'TNFSF18',
    '4-1BBL': 'TNFSF9',
    'CD27L': 'CD70',
    'CD30L': 'TNFSF8',
    'CD40L': 'CD40LG',
    'OX40L': 'TNFSF4',
    'FasL': 'FASLG',
    'RANKL': 'TNFSF11',
    
    # 生长因子及其他
    'TGF-beta1': 'TGFB1',
    'FGF-beta': 'FGF2',  # 碱性成纤维细胞生长因子
    'EGF': 'EGF',
    'VEGF': 'VEGFA',
    'HGF': 'HGF',
    'IGF-1': 'IGF1',
    'GDNF': 'GDNF',
    'PSPN': 'PSPN',
    'SCF': 'KITLG',      # 干细胞因子
    'FLT3L': 'FLT3LG',
    'TPO': 'THPO',       # 血小板生成素
    'EPO': 'EPO',
    'CT-1': 'CTF1',      # 心肌营养素-1
    'LIF': 'LIF',
    'OSM': 'OSM',
    'TSLP': 'TSLP',
    'ADSF': 'RETN',      # 抵抗素
    'Leptin': 'LEP',
    'Noggin': 'NOG',
    'Decorin': 'DCN',
    'PRL': 'PRL',
    
    # 补体片段 (片段无独立基因，映射到母体蛋白基因)
    'C3a': 'C3',
    'C5a': 'C5',
    
    # 淋巴毒素复合物
    'LT-alpha2-beta1': 'LTA', # 异源三聚体，映射到主亚基
    'LT-alpha1-beta2': 'LTB'  # 异源三聚体，映射到主亚基
}

In [6]:
adata.obs[condition_keys] = (
    adata.obs[condition_keys]
    .astype(str)                            # 1. 解除 category 限制，转为普通字符串
    .map(lambda x: gene_fix_map.get(x, x))  # 2. 替换：如果在字典里就替换，不在（如 PBS）就保持原样 x
    .astype('category')                     # 3. 重新转回 category 类型，节省内存并加快后续计算
)

In [7]:
adata.obs[control_key] = (adata.obs[condition_keys] == control_name)
gene_list = adata[adata.obs[control_key]==False].obs[condition_keys].unique()

In [8]:
gene_list

['CSF1', 'IL2', 'CTF1', 'IFNG', 'C5', ..., 'IL24', 'TNFSF13', 'VEGFA', 'TNFSF13B', 'EPO']
Length: 90
Categories (90, object): ['C3', 'C5', 'CD40LG', 'CD70', ..., 'TNFSF8', 'TNFSF9', 'TSLP', 'VEGFA']

In [9]:
df = pd.DataFrame(gene_list, columns=["gene"])

# 2. 保存：header=False 去掉表头，index=False 去掉左侧数字索引
df.to_csv('./data/raw/pbmc_data_target_genes.csv', index=False, encoding='utf-8-sig')

In [10]:
%run ./src/external/make_embedding.py --genes './data/raw/pbmc_data_target_genes.csv' --out "./data/processed/pbmc_data_target_genes_embedding.pkl"

Fetched: 500 / 763
Fetched: 321 / 321
Fetched: 254 / 254
Fetched: 500 / 533
Fetched: 500 / 837
Fetched: 208 / 208
Fetched: 500 / 793
Fetched: 240 / 240
Retrying in 3s
Fetched: 140 / 140
Fetched: 500 / 640
Retrying in 3s
Fetched: 381 / 381
Retrying in 3s
Fetched: 44 / 44
Retrying in 3s
Retrying in 3s
Retrying in 3s
Fetched: 500 / 773
Fetched: 6 / 6
Retrying in 3s
Fetched: 189 / 189
Retrying in 3s
Fetched: 110 / 110
Retrying in 3s
Fetched: 378 / 378
Retrying in 3s
Fetched: 118 / 118
Retrying in 3s
Fetched: 148 / 148
Retrying in 3s
Fetched: 500 / 518
Retrying in 3s
Retrying in 3s
Fetched: 428 / 428
Retrying in 3s
Fetched: 334 / 334
Retrying in 3s
Retrying in 3s
Fetched: 500 / 623
Retrying in 3s
Fetched: 319 / 319
Retrying in 3s
Fetched: 356 / 356
Retrying in 3s
Fetched: 81 / 81
Fetched: 33 / 33
Retrying in 3s
Fetched: 244 / 244
Retrying in 3s
Fetched: 242 / 242
Retrying in 3s
Retrying in 3s
Fetched: 500 / 885
Retrying in 3s
Retrying in 3s
Fetched: 500 / 526
Retrying in 3s
Fetched: 234 / 2

  0%|          | 0/90 [00:00<?, ?it/s]

1


  1%|          | 1/90 [00:01<01:51,  1.25s/it]

1


...

1


 99%|█████████▉| 89/90 [00:52<00:00,  1.52it/s]

1


100%|██████████| 90/90 [00:52<00:00,  1.71it/s]

2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
{'CSF1': tensor([ 0.0800, -0.0720,  0.0787,  ..., -0.0766,  0.1055,  0.0799]), 'IL2': tensor([ 0.0636, -0.0291,  0.0171,  ..., -0.0324,  0.0169, -0.0682]), 'CTF1': tensor([-0.0615, -0.0161,  0.1083,  ..., -0.0825,  0.1676,  0.0763]), 'IFNG': tensor([ 0.0832,  0.0023, -0.0182,  ..., -0.0105, -0.1107, -0.0318]), 'C5': tensor([ 0.0034, -0.0319, -0.0455,  ..., -0.0892, -0.0949,  0.0801]), 'IL17F': tensor([ 0.0420, -0.0685, -0.0266,  ..., -0.0130, -0.0676, -0.0753]), 'HGF': tensor([-0.0026, -0.0515,  0.0166,  ..., -0.0981, -0.0500,  0.0108]), 'IL36RN': tensor([-0.0742, -0.0294, -0.1317,  ..., -0.1255,  0.0350,  0.0780]), 'TNFSF18': tensor([ 0.0488, -0.0328, -0.0606,  ..., -0.1855, -0.0753,  0.0795]), 'IL10': tensor([-0.0397, -0.0034, -0.0165,  ..., -0.0843,  0.0288,  0.0610]), 'IL1RN': tensor([-0.0804,  0.0062, -0

In [11]:
import pandas as pd
from UniProtMapper import ProtMapper
from tqdm import tqdm

# 你的基因列表

def check_genes(genes):
    mapper = ProtMapper()
    results = []
    
    print("开始检查基因匹配情况...")
    for gene in tqdm(genes):
        try:
            # 执行查询
            result, failed = mapper.get(ids=gene, from_db="Gene_Name", to_db="UniProtKB")
            
            # 模拟原脚本的筛选条件
            filtered = result[(result['Organism'] == "Homo sapiens (Human)") & (result['Reviewed'] == "reviewed")]
            
            if filtered.empty:
                status = "❌ 找不到匹配 (No Reviewed Human Entry)"
            else:
                entry = filtered.iloc[0]['Entry']
                status = f"✅ 匹配成功: {entry}"
        except Exception as e:
            status = f"⚠️ 查询出错: {str(e)}"
            
        results.append({"Gene": gene, "Status": status})
    
    return pd.DataFrame(results)

# 运行检查
# genes = [gene_fix_map.get(g, g) for g in gene_list]
report_df = check_genes(gene_list)

# 输出结果：只看失败的基因
failed_genes = report_df[report_df['Status'].str.contains("❌|⚠️")]
print(f"\n检查完成！共有 {len(failed_genes)} 个基因匹配失败。")
if not failed_genes.empty:
    print(failed_genes)

开始检查基因匹配情况...


  1%|          | 1/90 [00:04<06:59,  4.72s/it]

Fetched: 500 / 763


  2%|▏         | 2/90 [00:08<05:50,  3.99s/it]

Fetched: 321 / 321


...

 10%|█         | 9/90 [00:33<04:52,  3.61s/it]

Fetched: 140 / 140


 11%|█         | 10/90 [00:37<04:59,  3.74s/it]

Fetched: 500 / 640


KeyboardInterrupt: 